# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata  # This is a DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This section helps us understand the dataset's structure, what recordsets and fields are available, and which `@id` values are used for programmatic access.

In [ ]:
# List all record sets (@id and name)
print("Available record sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  name: {rs['name']}")
    if 'description' in rs:
        print(f"  description: {rs['description']}")

# For each record set, show its fields (@id)
print("\nFields in each record set:")
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    fields = rs.get('field', [])
    # The field may be a single dict or list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', None)
        else:
            field_id = field
        print(f"  - field @id: {field_id}")

## 3. Data Extraction
Load data from selected record set(s) into pandas DataFrames for analysis. We'll use the record set and field `@id`s obtained above.

> Note: Replace `<record_set_id>` with the relevant record set `@id` you want to explore. Similarly, list the desired record set `@id`s in the `record_sets_ids` list below.

In [ ]:
# Example: extracting all record sets

# List all record set @id's
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}  # Map from record set @id to DataFrame

print("Loading records for each record set...")

for record_set_id in record_sets_ids:
    print(f"- Loading: {record_set_id}")
    records_itr = dataset.records(record_set=record_set_id)
    records = list(records_itr)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded shape: {df.shape}")
    else:
        print("  No records found.")

# For demonstration, pick one of the record sets (first if available)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular data loaded. Please check the dataset structure.")

## 4. Exploratory Data Analysis (EDA)
Let's process and examine the data. We'll select a numeric field (by `@id` as listed above), filter records, normalize values, and group by another field.

> Update `numeric_field_id` and `group_field_id` as appropriate for your data structure. All fields should be referenced by their full `@id`.

In [ ]:
# Example field IDs -- update these based on the previous overview output
# Let's try to guess typical field ids for age and sex, if available, else just print available columns
selected_df = dataframes.get(main_record_set_id)

if selected_df is not None:
    print(f"\nAvailable columns in '{main_record_set_id}':")
    for col in selected_df.columns:
        print(f"- {col}")

    # Guess potential numeric and groupable field IDs (for demonstration)
    # Try to pick age or similar, and group by sex or anatomical location if available
    numeric_field_id = None
    group_field_id = None
    for col in selected_df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
        if ('sex' in col.lower()) or ('gender' in col.lower()):
            group_field_id = col
        if ('location' in col.lower()):
            group_field_id = col

    if not numeric_field_id:
        # fallback: choose the first numeric column
        for col in selected_df.columns:
            if pd.api.types.is_numeric_dtype(selected_df[col]):
                numeric_field_id = col
                break

    if not numeric_field_id:
        print("No numeric field found for EDA.")
    else:
        threshold = selected_df[numeric_field_id].mean() if selected_df[numeric_field_id].dtype.kind in 'fi' else 0
        filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (mean value):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found in columns.")
else:
    print("No records DataFrame found. Please check previous steps.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Here we plot the normalized numeric field and the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Visualize the data if available
if selected_df is not None and numeric_field_id and numeric_field_id in selected_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(selected_df[numeric_field_id], kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in selected_df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=selected_df[group_field_id], y=selected_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and analyze a FAIR² colorectal cancer dataset using the `mlcroissant` library. By referencing entity `@id`s throughout, we ensured programmatic consistency and clear reproducibility. We provided record set overviews, loaded data, performed normalization and grouping, and visualized results. Further domain-specific analyses can be performed as desired on any available fields using their `@id`s.

> For additional operations or advanced analytics, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).